In [ ]:
import os
import math
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType, NumericType

print("Done")

In [ ]:
BUCKET_NAME = "hcdr"
RAW_PATH     = "s3://{}/raw/".format(BUCKET_NAME)
TRUSTED_PATH = "s3://{}/trusted/".format(BUCKET_NAME)
REFINED_PATH = "s3://{}/refined/eda/".format(BUCKET_NAME)

RAW_DB = "raw_db"
TRUSTED_DB = "hcdr_trusted"

USE_GLUE_CATALOG = True
ALLOW_S3_FALLBACK = False
SAMPLE_FRACTION = 0.03

PATHS = {
    "application_train"      : RAW_PATH + "application_train/application_train.csv",
    "bureau"                 : RAW_PATH + "bureau/bureau.csv",
    "bureau_balance"         : RAW_PATH + "bureau_balance/bureau_balance.csv",
    "previous_application"   : RAW_PATH + "previous_application/previous_application.csv",
    "pos_cash_balance"       : RAW_PATH + "pos_cash_balance/POS_CASH_balance.csv",
    "credit_card_balance"    : RAW_PATH + "credit_card_balance/credit_card_balance.csv",
    "installments_payments"  : RAW_PATH + "installments_payments/installments_payments.csv",
}

print("Configuración lista")
print("RAW_PATH     :", RAW_PATH)
print("TRUSTED_PATH :", TRUSTED_PATH)
print("REFINED_PATH :", REFINED_PATH)
print("RAW_DB       :", RAW_DB)
print("USE_GLUE_CATALOG:", USE_GLUE_CATALOG)

In [ ]:
try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext

    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    print("Spark inicializado")
except Exception as e:
    print("No se pudo usar GlueContext")
    spark = (
        SparkSession.builder
        .appName("EDA_HomeCreditRisk_SparkSQL_AWS")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.shuffle.partitions", "120")
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


In [ ]:
def normalize_column_names(df):
    for old in df.columns:
        new = old.strip().lower().replace(" ", "_")
        if old != new:
            df = df.withColumnRenamed(old, new)
    return df


def load_table(view_name, catalog_table=None, s3_path=None):
    catalog_table = catalog_table or view_name

    if USE_GLUE_CATALOG:
        try:
            full_name = "{}.{}".format(RAW_DB, catalog_table)
            df = spark.table(full_name)
            df = normalize_column_names(df)
            df.createOrReplaceTempView(view_name)
            print("\tVista SparkSQL creada: {} | {:,} filas × {} columnas".format(view_name, df.count(), len(df.columns)))
            return df
        except Exception as e:
            print("No se pudo leer desde Glue Catalog:", catalog_table)
            print("\tMotivo:", str(e)[:250])
            if not ALLOW_S3_FALLBACK:
                raise e

    if s3_path is None:
        raise ValueError("No hay ruta S3 para fallback de {}".format(view_name))

    print("\tLeyendo CSV desde S3:", s3_path)
    df = spark.read.csv(s3_path, header=True, inferSchema=True, nullValue="", nanValue="NaN")
    df = normalize_column_names(df)
    df.createOrReplaceTempView(view_name)
    print("\tVista SparkSQL creada: {} | {:,} filas × {} columnas".format(view_name, df.count(), len(df.columns)))
    return df


def sql(query, rows=20, truncate=False):
    result = spark.sql(query)
    result.show(rows, truncate=truncate)
    return result


def guardar_spark_en_refined(df, nombre):
    out_path = REFINED_PATH + nombre.strip("/") + "/"
    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", "true")
          .csv(out_path)
    )
    print("\tGuardado en refined:", out_path)


def guardar_parquet_trusted(df, nombre):
    out_path = TRUSTED_PATH + nombre.strip("/") + "/"
    (
        df.write
          .mode("overwrite")
          .option("compression", "snappy")
          .parquet(out_path)
    )
    print("\tGuardado en trusted:", out_path)


def plot_bar_from_pandas(pdf, x, y, title, xlabel=None, ylabel=None, rotation=0):
    if pdf.empty:
        print("\tDataFrame vacío. No se genera gráfico.")
        return
    plt.figure(figsize=(10, 4))
    plt.bar(pdf[x].astype(str), pdf[y])
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.xticks(rotation=rotation, ha="right" if rotation else "center")
    plt.tight_layout()
    plt.show()


def plot_hbar_from_pandas(pdf, x, y, title, xlabel=None, ylabel=None):
    if pdf.empty:
        print("\tDataFrame vacío. No se genera gráfico.")
        return
    plt.figure(figsize=(10, max(4, len(pdf) * 0.35)))
    plt.barh(pdf[y].astype(str), pdf[x])
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


def build_null_sql(view_name, columns):
    selects = []
    stack_entries = []
    for i, c in enumerate(columns):
        alias = "n_{}".format(i)
        selects.append("SUM(CASE WHEN `{}` IS NULL THEN 1 ELSE 0 END) AS {}".format(c, alias))
        stack_entries.append("'{}', {}".format(c, alias))

    query = """
    WITH base AS (
        SELECT
            COUNT(*) AS total,
            {selects}
        FROM {view_name}
    )
    SELECT
        columna,
        nulos,
        ROUND(100.0 * nulos / total, 2) AS pct_nulos
    FROM base
    LATERAL VIEW stack({n}, {stack_entries}) s AS columna, nulos
    ORDER BY pct_nulos DESC, nulos DESC
    """.format(
        selects=",\n            ".join(selects),
        view_name=view_name,
        n=len(columns),
        stack_entries=", ".join(stack_entries)
    )
    return query


def value_counts_sql(view_name, column_name, target_col="target"):
    return """
    SELECT
        '{col}' AS variable,
        CAST(`{col}` AS STRING) AS valor,
        COUNT(*) AS total,
        SUM(CASE WHEN `{target}` = 1 THEN 1 ELSE 0 END) AS defaults,
        ROUND(100.0 * SUM(CASE WHEN `{target}` = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct
    FROM {view}
    WHERE `{col}` IS NOT NULL
      AND `{target}` IN (0, 1)
    GROUP BY `{col}`
    ORDER BY total DESC
    """.format(col=column_name, target=target_col, view=view_name)

print("Done")

In [ ]:
TABLES = {}

for name, path in PATHS.items():
    TABLES[name] = load_table(name, catalog_table=name, s3_path=path)

app = TABLES["application_train"]
bureau = TABLES["bureau"]
bureau_balance = TABLES["bureau_balance"]
previous_application = TABLES["previous_application"]
pos_cash_balance = TABLES["pos_cash_balance"]
credit_card_balance = TABLES["credit_card_balance"]
installments_payments = TABLES["installments_payments"]

print("\n\tTodas las tablas quedaron disponibles como vistas SparkSQL")
spark.sql("SHOW TABLES").show(truncate=False)

In [ ]:
inventory_rows = []

for nombre, df in TABLES.items():
    temp_view = "tmp_inv_{}".format(nombre)
    spark.sql("""
        SELECT
            '{nombre}' AS tabla,
            COUNT(*) AS filas
        FROM {nombre}
    """.format(nombre=nombre)).createOrReplaceTempView(temp_view)

inventory_query = " UNION ALL ".join([
    "SELECT * FROM tmp_inv_{}".format(nombre) for nombre in TABLES.keys()
])

inventario_sql = spark.sql(inventory_query)
inventario_sql.show(truncate=False)
guardar_spark_en_refined(inventario_sql, "inventario_raw")

In [ ]:
schema_rows = []
for nombre, df in TABLES.items():
    for field in df.schema.fields:
        schema_rows.append((nombre, field.name, str(field.dataType)))

schema_df = spark.createDataFrame(schema_rows, ["tabla", "columna", "tipo"])
schema_df.createOrReplaceTempView("schema_inventario")

sql("""
SELECT tabla, columna, tipo
FROM schema_inventario
ORDER BY tabla, columna
""", rows=200, truncate=False)

guardar_spark_en_refined(schema_df, "schema_inventario")

In [ ]:
sql("""
SELECT COUNT(*) AS total_filas
FROM application_train
""")

In [ ]:
target_dist = sql("""
SELECT
    target,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM application_train
WHERE target IN (0, 1)
GROUP BY target
ORDER BY target
""")

guardar_spark_en_refined(target_dist, "target_distribution")

target_pd = target_dist.toPandas()
plot_bar_from_pandas(
    target_pd,
    x="target",
    y="total",
    title="Distribución de TARGET",
    xlabel="TARGET",
    ylabel="Cantidad"
)

print(target_pd)

In [ ]:
nulls_app = spark.sql(build_null_sql("application_train", app.columns))
nulls_app.show(50, truncate=False)

guardar_spark_en_refined(nulls_app, "nulos_application_train")

nulls_app_pd = nulls_app.limit(30).toPandas()
plot_hbar_from_pandas(
    nulls_app_pd,
    x="pct_nulos",
    y="columna",
    title="Top 30 columnas con mayor porcentaje de nulos — application_train",
    xlabel="% de nulos",
    ylabel="Columna"
)

In [ ]:
NULL_DROP_THRESHOLD = 0.40

def clasify_columns(df, umbral=NULL_DROP_THRESHOLD):
    total = df.count()
    cols_drop, cols_num, cols_cat = [], [], []

    for field in df.schema.fields:
        col = field.name
        n_nulls = df.filter(F.col(col).isNull()).count()
        pct = n_nulls / total

        if pct >= umbral:
            cols_drop.append((col, round(pct * 100, 1)))
        elif isinstance(field.dataType, NumericType):
            cols_num.append(col)
        else:
            if n_nulls > 0:
                cols_cat.append(col)

    return cols_drop, cols_num, cols_cat


def fill_with_medians(df, columnas):
    if not columnas:
        return df
    medianas = df.approxQuantile(columnas, [0.5], 0.001)
    fill_map = {
        col: (med[0] if med[0] is not None else 0.0)
        for col, med in zip(columnas, medianas)
    }
    return df.fillna(fill_map)


def fill_with_unknowns(df, columnas, valor="UNKNOWN"):
    if not columnas:
        return df
    return df.fillna({col: valor for col in columnas})


def process_table(df, nombre, cols_excluir_drop=None, tratamientos_especiales=None):
    cols_excluir_drop = cols_excluir_drop or []
    print("\n" + "="*60)
    print("PREPARANDO:", nombre.upper())
    print("Filas originales: {:,}  |  Columnas: {}".format(df.count(), len(df.columns)))

    cols_drop, cols_num, cols_cat = clasify_columns(df)

    cols_drop_reales = [(c, p) for c, p in cols_drop if c not in cols_excluir_drop]
    nombres_drop = [c for c, _ in cols_drop_reales]

    if nombres_drop:
        print("\n  [DROP] Columnas eliminadas por ≥{}% nulos:".format(int(NULL_DROP_THRESHOLD*100)))
        for c, p in cols_drop_reales:
            print("         - {} ({:.1f}%)".format(c, p))
        df = df.drop(*nombres_drop)

    cols_num = [c for c in cols_num if c in df.columns]
    if cols_num:
        print("\n  [IMPUTAR-NUM] {} columnas numéricas → mediana".format(len(cols_num)))
        df = fill_with_medians(df, cols_num)

    cols_cat = [c for c in cols_cat if c in df.columns]
    if cols_cat:
        print("  [IMPUTAR-CAT] {} columnas categóricas → 'UNKNOWN'".format(len(cols_cat)))
        df = fill_with_unknowns(df, cols_cat)

    if tratamientos_especiales:
        for col_name, (cond_col, val_nuevo, val_reemplazo) in tratamientos_especiales.items():
            if col_name in df.columns:
                print("  [ESPECIAL] {}: {} → {}".format(col_name, val_nuevo, val_reemplazo))
                df = df.withColumn(
                    col_name,
                    F.when(F.col(col_name) == val_nuevo, val_reemplazo).otherwise(F.col(col_name))
                )

    print("\n  Filas finales: {:,}  |  Columnas finales: {}".format(df.count(), len(df.columns)))

    guardar_parquet_trusted(df, nombre)
    return df


print("Done")

In [ ]:
app_prep = app.withColumn(
    "days_employed",
    F.when(F.col("days_employed") == 365243, None).otherwise(F.col("days_employed"))
).alias("app_prep")

app_prep = app_prep.withColumn(
    "flag_days_employed_anomalo",
    F.when(F.col("days_employed").isNull(), 1).otherwise(0)
)

app_trusted = process_table(
    df=app_prep,
    nombre="application_train",
    cols_excluir_drop=["sk_id_curr", "target"]
)
app_trusted.createOrReplaceTempView("application_train_trusted")
print("Done")

In [ ]:
bureau_trusted = process_table(
    df=bureau,
    nombre="bureau",
    cols_excluir_drop=["sk_id_curr", "sk_id_bureau"]
)

bureau_trusted.createOrReplaceTempView("bureau_trusted")
print("Done")

In [ ]:
bureau_balance_trusted = process_table(
    df=bureau_balance,
    nombre="bureau_balance",
    cols_excluir_drop=["sk_id_bureau"]
)

bureau_balance_trusted.createOrReplaceTempView("bureau_balance_trusted")
print("Done")

In [ ]:
prev_app_clean = previous_application.withColumn(
    "days_first_drawing",
    F.when(F.col("days_first_drawing") == 365243, None).otherwise(F.col("days_first_drawing"))
).withColumn(
    "days_first_due",
    F.when(F.col("days_first_due") == 365243, None).otherwise(F.col("days_first_due"))
).withColumn(
    "days_last_due_1st_version",
    F.when(F.col("days_last_due_1st_version") == 365243, None).otherwise(F.col("days_last_due_1st_version"))
).withColumn(
    "days_last_due",
    F.when(F.col("days_last_due") == 365243, None).otherwise(F.col("days_last_due"))
).withColumn(
    "days_termination",
    F.when(F.col("days_termination") == 365243, None).otherwise(F.col("days_termination"))
)

prev_app_trusted = process_table(
    df=prev_app_clean,
    nombre="previous_application",
    cols_excluir_drop=["sk_id_curr", "sk_id_prev"]
)

prev_app_trusted.createOrReplaceTempView("previous_application_trusted")
print("Done")

In [ ]:
pos_cash_trusted = process_table(
    df=pos_cash_balance,
    nombre="pos_cash_balance",
    cols_excluir_drop=["sk_id_curr", "sk_id_prev"]
)

pos_cash_trusted.createOrReplaceTempView("pos_cash_balance_trusted")
print("Done")

In [ ]:
credit_card_trusted = process_table(
    df=credit_card_balance,
    nombre="credit_card_balance",
    cols_excluir_drop=["sk_id_curr", "sk_id_prev"]
)

credit_card_trusted.createOrReplaceTempView("credit_card_balance_trusted")
print("Done")

In [ ]:
installments_trusted = process_table(
    df=installments_payments,
    nombre="installments_payments",
    cols_excluir_drop=["sk_id_curr", "sk_id_prev"]
)

installments_trusted.createOrReplaceTempView("installments_payments_trusted")
print("Done")

In [ ]:
TRUSTED_TABLES = {
    "application_train"     : app_trusted,
    "bureau"                : bureau_trusted,
    "bureau_balance"        : bureau_balance_trusted,
    "previous_application"  : prev_app_trusted,
    "pos_cash_balance"      : pos_cash_trusted,
    "credit_card_balance"   : credit_card_trusted,
    "installments_payments" : installments_trusted,
}

print("\n{:<30} {:>12} {:>12} {:>12}".format("Tabla", "Filas", "Columnas", "Nulos Total"))
print("-" * 70)

resumen_rows = []

for nombre, df in TRUSTED_TABLES.items():
    filas = df.count()
    columnas = len(df.columns)

    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    null_counts = df.select(null_exprs).collect()[0]
    total_nulos = sum(null_counts)

    print("{:<30} {:>12,} {:>12} {:>12,}".format(nombre, filas, columnas, total_nulos))
    resumen_rows.append((nombre, filas, columnas, total_nulos))

resumen_df = spark.createDataFrame(resumen_rows, ["tabla", "filas", "columnas", "nulos_residuales"])
guardar_spark_en_refined(resumen_df, "quality_resume_trusted")

print("\nResumen de calidad guardado en refined/quality_resume_trusted")